# Testing Labeling process

Expanding Z-score within this selected address

In [1]:
import os
import pandas as pd

In [2]:
daily_filtered = pd.read_parquet("../data/processed/chain/scam_daily_cleaned.parquet")

output_folder = "../data/per_address"
os.makedirs(output_folder, exist_ok=True)

key_features = [
    'normal_sent_cnt', 'normal_recv_cnt', 'normal_total_cnt',
    'eth_sent_sum', 'eth_recv_sum', 'eth_net_flow',
    'uniq_peers_cnt', 'sessions_cnt', 'active_span_min', 'burst_max_tx_5m'
]

In [4]:
threshold = 2.0
target_address = "0xb7bfcdc3a2aa2af3fe653c9e8a19830977e1993c"
eps = 1e-6

daily_filtered["day"] = pd.to_datetime(daily_filtered["day"])

addr_df = (
    daily_filtered[daily_filtered["address"] == target_address]
    .sort_values("day")
    .reset_index(drop=True)
    .copy()
)

z_cols = []
for feature in key_features:
    exp_mean = addr_df[feature].expanding(min_periods=1).mean()
    exp_std  = addr_df[feature].expanding(min_periods=1).std().fillna(0).replace(0, eps)

    addr_df[f"{feature}_z"] = (addr_df[feature] - exp_mean) / exp_std
    z_cols.append(f"{feature}_z")

addr_df["combined_z_score"] = addr_df[z_cols].abs().max(axis=1)
addr_df["is_anomalous"] = (addr_df["combined_z_score"] > threshold).astype(int)

output_path = os.path.join(output_folder, f"{target_address}_z_score.xlsx")
addr_df[["day"] + key_features + ["combined_z_score", "is_anomalous"]].to_excel(output_path, index=False)

print("Saved:", output_path)

Saved: ../data/per_address\0xb7bfcdc3a2aa2af3fe653c9e8a19830977e1993c_z_score.xlsx
